# WorldSim DGX Spark QLoRA Smoke Run

Thin Jupyter wrapper around the shared `training.lib.qlora_smoke` API. Use this notebook to verify the DGX Spark kernel, run `preflight` / `smoke` / `longer_smoke`, and inspect post-run JSON structure without duplicating trainer logic.


In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd()
repo_marker = cwd / "training/lib/qlora_smoke.py"
assert repo_marker.exists(), f"Run this notebook from the repo root. cwd={cwd}"
{
    "python_executable": sys.executable,
    "cwd": str(cwd),
    "repo_marker": str(repo_marker),
}


## Run Mode

Edit only this cell to switch between `preflight`, `smoke`, and `longer_smoke`. The shared helper resolves the final step/sample counts so the notebook stays thin.


In [ ]:
from training.lib.qlora_smoke import resolve_notebook_run_mode

RUN_MODE = "smoke"  # preflight | smoke | longer_smoke
RUN_ID = None  # set an explicit string to reuse a fixed output folder

resolved_run = resolve_notebook_run_mode(RUN_MODE, run_id=RUN_ID)
RUN_MODE = resolved_run["run_mode"]
RUN_ID = resolved_run["run_id"]
MAX_STEPS = resolved_run["max_steps"]
MAX_TRAIN_SAMPLES = resolved_run["max_train_samples"]
MAX_EVAL_SAMPLES = resolved_run["max_eval_samples"]

resolved_run


## Environment Visibility

Run this before training. If CUDA or bitsandbytes looks wrong here, stop and switch to the correct DGX Spark kernel.


In [ ]:
from training.lib.qlora_smoke import get_environment_summary

environment = get_environment_summary()
torch_info = environment.get("torch", {})
bnb_info = environment.get("bitsandbytes", {})
{
    "python_executable": environment.get("python", {}).get("executable"),
    "cwd": environment.get("cwd"),
    "torch_version": torch_info.get("version"),
    "torch_cuda_version": torch_info.get("cuda_version"),
    "torch_cuda_available": torch_info.get("cuda_available"),
    "gpu_count": torch_info.get("cuda_device_count", 0),
    "gpu_names": torch_info.get("cuda_device_names", []),
    "bitsandbytes_available": bnb_info.get("available"),
    "bitsandbytes_version": bnb_info.get("version"),
    "environment_summary": environment,
}


## Strict True-QLoRA Preflight

This uses the same shared runtime detection as the CLI path and hard-fails before training if true CUDA 4-bit QLoRA cannot be satisfied.


In [ ]:
from training.lib.qlora_smoke import get_true_qlora_preflight

preflight = get_true_qlora_preflight()
assert preflight["ok"], preflight["blocker_reason"]
preflight


## Config

This cell builds `SmokeRunConfig` from the resolved run mode. Keep edits here if you need to override model or batch sizing.


In [ ]:
from pathlib import Path

from training.lib.qlora_smoke import DEFAULT_MODEL_NAME, SmokeRunConfig

OUTPUT_DIR = Path("outputs/smoke_cuda_notebook/worldsim-v31-mix-v1") / RUN_ID

config = SmokeRunConfig(
    model_name=DEFAULT_MODEL_NAME,
    train_file=Path("data/training/worldsim-v31-mix-v1/train_converted.jsonl"),
    dev_file=Path("data/training/worldsim-v31-mix-v1/dev_converted.jsonl"),
    output_dir=OUTPUT_DIR,
    max_steps=MAX_STEPS,
    max_train_samples=MAX_TRAIN_SAMPLES,
    max_eval_samples=MAX_EVAL_SAMPLES,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=1,
    learning_rate=2e-4,
    require_qlora=True,
    seed=42,
)

config.to_dict()


## Run Smoke Training

This cell stays intentionally thin and calls the shared API directly.


In [ ]:
import time

from training.lib.qlora_smoke import run_smoke_or_raise

started_at = time.perf_counter()
result = run_smoke_or_raise(config)
elapsed_seconds = round(time.perf_counter() - started_at, 2)
{
    "elapsed_seconds": elapsed_seconds,
    "result": result.to_dict(),
}


## Inspect Run Artifacts

These cells group the key JSON artifacts so you can quickly see what configuration ran, whether true QLoRA stayed active, and what the train/eval metrics look like.


In [ ]:
from training.lib.qlora_smoke import load_json_artifact, preview_metrics

run_config = load_json_artifact(result.output_dir, "run_config.json")
summary = load_json_artifact(result.output_dir, "summary.json")
metrics = preview_metrics(result.output_dir)

{
    "run_config": {
        "model_name": run_config.get("config", {}).get("model_name"),
        "max_steps": run_config.get("config", {}).get("max_steps"),
        "max_train_samples": run_config.get("config", {}).get("max_train_samples"),
        "max_eval_samples": run_config.get("config", {}).get("max_eval_samples"),
        "require_qlora": run_config.get("config", {}).get("require_qlora"),
        "output_dir": run_config.get("output_dir"),
    },
    "summary": {
        "status": summary.get("status"),
        "used_true_qlora": summary.get("used_true_qlora"),
        "train_loss": summary.get("train_loss"),
        "eval_loss": summary.get("eval_loss"),
        "finite_losses": summary.get("finite_losses"),
        "train_rows": summary.get("train_rows"),
        "eval_rows": summary.get("eval_rows"),
    },
    "metrics": {
        "train_runtime": metrics.get("train_runtime"),
        "train_steps_per_second": metrics.get("train_steps_per_second"),
        "train_loss": metrics.get("train_loss"),
        "eval_loss": metrics.get("eval_loss"),
        "eval_runtime": metrics.get("eval_runtime"),
    },
}


## Sample Generation Structure Inspection

Use the shared sample-generation helpers to separate raw parse failures, recoverable fenced JSON, truly malformed JSON, and enum drift.


In [ ]:
from training.lib.qlora_smoke import load_sample_generations, summarize_sample_generations

samples = load_sample_generations(result.output_dir)
sample_summary = summarize_sample_generations(samples)

{
    "raw_parseable_json": sample_summary.get("raw_parseable_json"),
    "fenced_json": sample_summary.get("fenced_json"),
    "fence_stripped_parseable_json": sample_summary.get("fence_stripped_parseable_json"),
    "recoverable_fenced_json": sample_summary.get("recoverable_fenced_json"),
    "malformed_json": sample_summary.get("malformed_json"),
    "enum_drift_total": sample_summary.get("enum_drift_total"),
    "classifications": sample_summary.get("classifications"),
    "failure_categories": sample_summary.get("failure_categories"),
}


In [ ]:
sample_summary.get("recoverable_examples", [])


In [ ]:
samples[:3]


## Final Operational Judgment

This is the compact status dict to use for go/no-go decisions after a smoke run.


In [ ]:
from training.lib.qlora_smoke import build_operational_judgment

final_status = build_operational_judgment(summary, sample_summary, output_dir=result.output_dir)
{
    "final_status": final_status,
    "recommended_next_action": final_status.get("recommended_next_action"),
}


## Next Recommended Config

If the current run is stable, use this next conservative config for the next escalation step.


In [ ]:
from training.lib.qlora_smoke import recommended_next_smoke_config

next_run = recommended_next_smoke_config()
{
    "current_run_mode": RUN_MODE,
    "current_output_dir": str(OUTPUT_DIR),
    "suggested_next_config": next_run,
}
